# Xử lý các hashtag trong cột 'video_title'

In [1]:
import pandas as pd
import re

# 1. Đọc file dữ liệu của bạn (thay tên file cho phù hợp nếu cần)
input_file = "combined_data.csv"
output_file = "silver1.csv"

print(f"Đang đọc dữ liệu từ {input_file}...")
df = pd.read_csv(input_file)

# 2. Xóa Hashtag trong cột video_title
# Regex r'#\w+' sẽ tìm dấu '#' và tất cả các chữ cái/số dính liền sau đó để thay bằng chuỗi rỗng
df['video_title'] = df['video_title'].astype(str).str.replace(r'#\w+', '', regex=True)

# 3. Dọn dẹp khoảng trắng thừa
# Khi xóa hashtag xong, câu có thể bị dư khoảng trắng (ví dụ: "Tiêu đề   hay" -> "Tiêu đề hay")
df['video_title'] = df['video_title'].str.replace(r'\s+', ' ', regex=True).str.strip()

# 4. Lưu lại file mới
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"✅ Đã xóa sạch hashtag ở cột video_title và lưu vào {output_file}!")

# In thử 5 dòng đầu tiên để kiểm tra
print("\nKết quả sau khi xóa hashtag:")
print(df['video_title'].head())

Đang đọc dữ liệu từ combined_data.csv...
✅ Đã xóa sạch hashtag ở cột video_title và lưu vào silver1.csv!

Kết quả sau khi xóa hashtag:
0    Huyền Thoại Đa Tình Páp-Lô Pi-Cát-Xô
1    Huyền Thoại Đa Tình Páp-Lô Pi-Cát-Xô
2    Huyền Thoại Đa Tình Páp-Lô Pi-Cát-Xô
3    Huyền Thoại Đa Tình Páp-Lô Pi-Cát-Xô
4    Huyền Thoại Đa Tình Páp-Lô Pi-Cát-Xô
Name: video_title, dtype: object


In [2]:
df.to_csv('silver1.csv', index=False)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7978 entries, 0 to 7977
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   platform                7978 non-null   object
 1   video_id                7978 non-null   object
 2   video_url               7978 non-null   object
 3   video_title             7978 non-null   object
 4   video_core_content      7978 non-null   object
 5   video_target_sentiment  7978 non-null   object
 6   video_highlight         7978 non-null   object
 7   comment                 7975 non-null   object
dtypes: object(8)
memory usage: 498.8+ KB


# Gen commonsense


In [ ]:
import os
import json
import time
import re
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm
from mistralai.client import Mistral

load_dotenv()
MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY", "Gqz0214aAWG9LekxQ9uMRGkQp65ZbtLA") 
MODEL_ID = "open-mistral-nemo" 
#open-mistral-nemo
#pixtral-12b-2409
#mistral-large-latest
client = Mistral(api_key=MISTRAL_API_KEY)

def get_mistral_batch_natural(batch_data):
    system_prompt = """<role>
Bạn là một TỪ ĐIỂN VĂN HÓA MẠNG và CHUYÊN GIA KIỂM CHỨNG SỰ THẬT (FACT-CHECKER) trung lập.
Nhiệm vụ của bạn là giải nghĩa từ lóng mạng và TÌM MỐI LIÊN KẾT giữa bình luận với Ngữ cảnh video.
</role>

<critical_rules>
ĐỂ TRÁNH RÒ RỈ DỮ LIỆU VÀ TRÁNH VIỆC BẠN "LƯỜI BIẾNG" TỪ CHỐI SUY LUẬN, BẮT BUỘC TUÂN THỦ:

1. NGỮ CẢNH LÀ CHÂN LÝ TỐI CAO (QUAN TRỌNG NHẤT): Mọi sự thật bạn cần đều nằm ở mục [Context]. Nếu [Context] nói nhân vật lăng nhăng, thì khi bình luận dùng từ lóng hiện đại ("trap boi", "fuck boy", "cắm sừng", "bắt cá nhiều tay"), BẠN PHẢI LIÊN KẾT TỪ LÓNG ĐÓ VỚI SỰ THẬT TRONG CONTEXT. Tuyệt đối KHÔNG ĐƯỢC trả lời "không có thông tin xác thực" cho các từ lóng mô tả tính cách.

2. GIỚI HẠN LỐI THOÁT HIỂM: CHỈ ĐƯỢC dùng câu "Bình luận hỏi về yếu tố ngoài lề..." ĐỐI VỚI các trường hợp hỏi xin nhạc, xin link, hoặc tự nói về bản thân (VD: "em mới 13 tuổi"). Nếu bình luận đang nói về nhân vật trong video, BẠN BẮT BUỘC PHẢI GIẢI THÍCH.

3. DANH SÁCH TỪ CẤM: Tuyệt đối KHÔNG sử dụng: "mỉa mai", "châm biếm", "hài hước", "đá đểu", "giễu nhại", "khen", "chê", "ý nói", "mục đích", "nhận xét cá nhân". 

4. CẤU TRÚC 2 CÂU ÉP BUỘC: 
   - Câu 1: Định nghĩa từ lóng, teencode, hoặc hình ảnh so sánh trong bình luận.
   - Câu 2: Trích dẫn sự thật từ [Context] (hoặc lịch sử) để chứng minh mối liên kết.
</critical_rules>

<examples>
[Input]
ID: 101 | Context: Video về thói trăng hoa của Picasso, có 23 từ trong tên khai sinh. | Comment: picasso là trap boi à
ID: 102 | Context: Video về thói trăng hoa của Picasso, có 23 từ trong tên khai sinh. | Comment: những người đó đều là nicki minaj nhé
ID: 103 | Context: Video về thói trăng hoa của Picasso, có 23 từ trong tên khai sinh. | Comment: oát đờ cái tên khai sinh
ID: 104 | Context: Video về thói trăng hoa của Picasso, có 23 từ trong tên khai sinh. | Comment: cho xin tên bài nhạc nền phút 2:30

[Output]
{
  "results": [
    {"id": 101, "commonsense": "'Trap boi' (trap boy) là từ lóng mạng chỉ những chàng trai chuyên lừa tình, đùa giỡn với tình cảm của người khác. Điều này tương đồng với thông tin trong ngữ cảnh về việc Picasso có thói trăng hoa, thay người yêu như thay áo và nhiều mối quan hệ phức tạp."},
    {"id": 102, "commonsense": "Trường phái hội họa Lập thể của Picasso thường vẽ phụ nữ với các tỷ lệ cơ thể bị bóp méo, cường điệu hóa. Sự so sánh này bắt nguồn từ việc nữ ca sĩ Nicki Minaj ngoài đời thực cũng nổi tiếng với vóc dáng phẫu thuật thẩm mỹ có tỷ lệ ba vòng cường điệu khác thường."},
    {"id": 103, "commonsense": "'Oát đờ' là cách đọc phiên âm tiếng Việt của cụm từ tiếng Anh 'What the...'. Thực tế, tên khai sinh đầy đủ của họa sĩ Picasso rất dài, bao gồm 23 từ ghép lại với nhau."},
    {"id": 104, "commonsense": "Bình luận hỏi về yếu tố âm nhạc ngoài lề. Không có thông tin xác thực về tên bài nhạc nền dựa trên ngữ cảnh được cung cấp."}
  ]
}
</examples>

<output_format>
BẠN CHỈ ĐƯỢC PHÉP TRẢ VỀ MỘT KHỐI JSON DUY NHẤT. Value của "commonsense" phải là chuỗi (string) và TUYỆT ĐỐI KHÔNG chứa từ khóa cảm xúc.
</output_format>

BÂY GIỜ, HÃY SUY LUẬN VÀ KIỂM CHỨNG SỰ THẬT CHO BATCH SAU:
"""


    user_prompt = "[Input]\n"
    for item in batch_data:
        user_prompt += f"ID: {item['id']}\nContext: {item['ctx']}\nComment: {item['cmt']}\n\n"
    user_prompt += "[Output (JSON)]:"

    try:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
        
        response = client.chat.complete(
            model=MODEL_ID,
            messages=messages,
            temperature=0.2, # Vừa đủ linh hoạt cho văn phong đời thường, vừa đủ chặt chẽ cho JSON
            response_format={"type": "json_object"}
        )
        
        content = response.choices[0].message.content
        if not content: return None
            
        try:
            # Xử lý an toàn nếu Pixtral lỡ sinh ra ký tự điều khiển ẩn
            content = re.sub(r'[\x00-\x1f]', '', content)
            result_json = json.loads(content, strict=False)
            return result_json.get("results", [])
        except json.JSONDecodeError:
            return None

    except Exception as e:
        print(f"Lỗi API: {str(e)}")
        return None
def process_dataset_batch_final(input_csv, output_csv, batch_size=10):
    # 1. LOGIC XÓA SẠCH DỮ LIỆU CŨ NGAY TỪ ĐẦU & CHỈ ĐỌC TỪ FILE REVERTED
    if os.path.exists(output_csv):
        print(f"\n[*] Tìm thấy file đang chạy dở: {output_csv}. Đang resume...")
        df = pd.read_csv(output_csv)
    else:
        # Nếu chưa có file chạy dở, ĐỌC THẲNG TỪ FILE GỐC CÓ EMOJI
        print(f"\n[*] Chạy lần đầu. Đang đọc file gốc: {input_csv}")
        df = pd.read_csv(input_csv)
        
        print("[*] Đang TIÊU DIỆT toàn bộ dữ liệu cột 'commonsense' cũ của Qwen...")
        # Ép cột commonsense thành rỗng hoàn toàn để Pixtral làm lại từ đầu
        df['commonsense'] = None 
        
        # Lưu đè ra file output ngay lập tức để làm mốc xuất phát trắng tinh
        df.to_csv(output_csv, index=False, encoding='utf-8-sig')
        print(f"[*] Đã tạo file giấy trắng chuẩn bị chạy Batch: {output_csv}")
    
    # Ensure 'commonsense' column exists
    if 'commonsense' not in df.columns:
        df['commonsense'] = None
        print(f"[*] Đã tạo file giấy trắng chuẩn bị chạy Batch: {output_csv}")

    # 2. LỌC NHỮNG DÒNG CHƯA LÀM HOẶC LỖI
    def needs_processing(x):
        val = str(x).strip()
        if val in ['nan', 'None', '']: return True
        if val.startswith("Lỗi"): return True
        if "[" in val or "]" in val: return True # Lọc luôn các dòng lỡ bị mảng
        return False

    pending_indices = df[df['commonsense'].apply(needs_processing)].index.tolist()
    print(f"Tổng số dòng đang chờ Pixtral xử lý: {len(pending_indices)} | Batch Size: {batch_size}")
    
    if not pending_indices:
        print("Mọi dữ liệu đã được làm xong 100%!")
        return

    # 3. CHẠY BATCH API
    for i in tqdm(range(0, len(pending_indices), batch_size), desc="Đang cày Batch (Pixtral)"):
        batch_indices = pending_indices[i:i+batch_size]
        
        batch_data = []
        for idx in batch_indices:
            ctx = str(df.at[idx, 'video_core_content'])
            cmt = str(df.at[idx, 'comment'])
            if ctx.lower() == 'nan': ctx = ''
            if cmt.lower() == 'nan': cmt = ''
            
            batch_data.append({"id": idx, "ctx": ctx, "cmt": cmt})
            
        # Gọi API Batch
        batch_results = get_mistral_batch_natural(batch_data)
        
        if batch_results:
            for res in batch_results:
                row_id = res.get("id")
                ans = res.get("commonsense", "")
                
                # Sửa lỗi nếu LLM ngoan cố trả về list
                if isinstance(ans, list): 
                    ans = " ".join([str(item) for item in ans])
                    
                if row_id is not None and row_id in df.index:
                    df.at[row_id, 'commonsense'] = str(ans).strip()
            
            tqdm.write(f"[Batch {i//batch_size}] Đã xử lý mượt mà {len(batch_results)} câu.")
        else:
            tqdm.write(f"[Lỗi Batch {i//batch_size}] Cú pháp hỏng, bỏ qua để lát chạy lại.")
        
        # Auto-save sau mỗi mẻ
        df.to_csv(output_csv, index=False, encoding='utf-8-sig')
        time.sleep(1) # Lách Rate Limit

    print(f"\n[HOÀN THÀNH ĐẠI THÀNH CÔNG] Dữ liệu chuẩn đã lưu tại: {output_csv}")

if __name__ == "__main__":
    # NGUỒN CẤP: Phải là file gốc chứa data có emoji, chưa qua Pixtral
    INPUT_FILE = "combined_data.csv" 
    
    # FILE XUẤT: File sạch sẽ hoàn toàn do Pixtral Batch sinh ra
    OUTPUT_FILE = "silver_nemo.csv" 
    
    # ĐỂ BATCH_SIZE = 5 LÀ AN TOÀN VÀ HIỆU QUẢ NHẤT
    BATCH_SIZE = 10
    
    if os.path.exists(INPUT_FILE):
        process_dataset_batch_final(INPUT_FILE, OUTPUT_FILE, batch_size=BATCH_SIZE)
    else:
        print(f"Lỗi: Không tìm thấy file gốc {INPUT_FILE}")


[*] Tìm thấy file đang chạy dở: silver1.csv. Đang resume...
[*] Đã tạo file giấy trắng chuẩn bị chạy Batch: silver1.csv
Tổng số dòng đang chờ Pixtral xử lý: 7978 | Batch Size: 10


Đang cày Batch (Pixtral):   0%|          | 0/798 [00:05<?, ?it/s]

[Batch 0] Đã xử lý mượt mà 10 câu.


Đang cày Batch (Pixtral):   0%|          | 1/798 [00:12<2:42:35, 12.24s/it]


KeyboardInterrupt: 